In [109]:
YEAR = 2025
TEAM = 'Cardinals'
LEAGUE = 'NL'
AB = 388
BB = 43
IBB = 1
HBP = 15
SINGLES = 78
DOUBLES = 13
TRIPLES = 0
HOMERS = 19
SF = 4

UBB = BB - IBB
PA = AB + BB - IBB + SF + HBP

In [110]:
import pandas as pds

league_weights = pds.read_csv('./data/fangraphs-guts-data-leaguewide-weights-by-year.csv')
year_weights = league_weights.loc[league_weights['Season'] == YEAR].reset_index(drop=True)

park_factors = pds.read_csv('./data/fangraphs-guts-data-park-factors-2025.csv')
team_park_factor = park_factors.loc[park_factors['Team'] == TEAM].reset_index(drop=True)
PF = team_park_factor['Basic (5yr)'] / 100

In [111]:
print(year_weights)
print("\nPARK FACTORS:")
print(team_park_factor)

   Season      wOBA  wOBAScale       wBB      wHBP       w1B      w2B  \
0    2025  0.313055    1.23167  0.691497  0.722289  0.882406  1.25191   

       w3B     wHR  runSB     runCS      R/PA      R/W      cFIP  
0  1.58446  2.0374    0.2 -0.409517  0.118158  9.77398  3.135136  

PARK FACTORS:
   Season       Team  Basic (5yr)        3yr        1yr          1B  \
0    2025  Cardinals    97.500134  99.491233  95.834076  101.070714   

          2B         3B         HR         SO         BB          GB  \
0  98.618948  88.620692  93.929237  97.267342  96.757519  100.878978   

           FB         LD        IFFB        FIP  
0  101.298451  99.400115  103.166687  97.936028  


In [112]:
wOBA = (
    UBB * year_weights['wBB'] + 
    HBP * year_weights['wHBP'] + 
    SINGLES * year_weights['w1B'] +
    DOUBLES * year_weights['w2B'] +
    TRIPLES * year_weights['w3B'] +
    HOMERS * year_weights['wHR']
) / (PA)

wRAA = ((wOBA - year_weights['wOBA']) / year_weights['wOBAScale']) * PA

wRC = (
        ((wOBA - year_weights['wOBA']) / year_weights['wOBAScale']) +
        year_weights['R/PA']
) * PA

"""
  Fangraphs is annoying vague about one particular number: the wRC+ formula
  page at https://library.fangraphs.com/offense/wrc/ says that one of the 
  necessary factors is "AL or NL wRC/PA excluding pitchers". But that number
  is not included in the Guts! page (https://www.fangraphs.com/tools/guts?type=cn)
  where all of the other factors are. I calculated it manuallu for 2025 and 
  hardcoded those numbers below. 
"""
AL_WRC_PA = 0.1178565558
NL_WRC_PA = 0.1184992854

WRC_PA = NL_WRC_PA if LEAGUE == 'NL' else AL_WRC_PA

"""
The wRC+ formula as documented by Fangraphs does not work. They claim:
wRC+ = (((wRAA/PA + League R/PA) + (League R/PA – Park Factor* League R/PA))/ (AL or NL wRC/PA excluding pitchers))*100
Unless I'm doing something terribly wrong, that formula as written returns nonsense.
For example, Ivan Herrera's 2025 wRC+ comes out to -9387.503348.

I rewrote the forumla to be conceptually correct, but now it doesn't match
with Fangraphs exactly. For example, Ivan Herrera's FG wRC+ is 137, while 
my calculation returns 138. But it's way closer than that junk above!
"""
wRC_park_factor = wRC / PF
wRC_pf_per_pa = wRC_park_factor / PA
wRCPlus = wRC_pf_per_pa / WRC_PA * 100
print(f"wRC+: {wRCPlus}")

wOBA: 0    0.364566
dtype: float64
wRAA: 0    18.778254
dtype: float64
wRC: 0    71.831196
dtype: float64
pf: 0    0.975001
Name: Basic (5yr), dtype: float64
wRC_park_factor: 0    73.672921
dtype: float64
wRC+: 0    138.466851
dtype: float64
